# Confidence V2 — Grid Search

**Amaç:** V2 metriklerinin (Rg, PAE, contact_recon, warmup, stall prevention) farklı cutoff'lardaki etkisini test etmek.

**Deneyler:**
1. Warmup etkisi (ilk N step'te gevşek cutoff)
2. Rg filtresi (yapı patlaması tespiti)
3. Max consecutive rejected (stall prevention)
4. Alpha decay on rejection
5. PAE cutoff
6. K=3 sample consensus
7. Contact reconstruction cutoff
8. OF3 distogram contact cutoff
9. Kombine (en iyi tekli sonuçlardan)

**Çalıştırma:** Cell 1 (setup) → Cell 2 (config) → Cell 3 (OF3 yükle) → Cell 4+ (deneyler)

## 1. Environment Setup (bir kere çalıştır)

In [ ]:
# ════════════════════════════════════════════════════
#  ONE-TIME SETUP — run once per runtime
# ════════════════════════════════════════════════════
import os, sys, shutil

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Always fresh clone ANM-openfold3
if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

# Clone and install OpenFold3
if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch

sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

# Ensure OF3 config dir exists
os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

# Pre-download OF3 model parameters
from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)
print(f'OF3 model parameters ready at {_param_dir}')

print('Environment setup complete.')

## 2. Config (her denemede değiştir)

In [ ]:
# ════════════════════════════════════════════════════
#  GRID SEARCH CONFIG
# ════════════════════════════════════════════════════

# ── PDB ────────────────────────────────────────────
INITIAL_PDB = "1AKE"          # Open conformation
TARGET_PDB  = "4AKE"          # Closed conformation
CHAIN_ID    = "A"

# ── Pipeline baseline ──────────────────────────────
N_STEPS              = 30       # steps per experiment run
COMBINATION_STRATEGY = "autostop"
Z_MIXING_ALPHA       = 0.7
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = "plus"

# ── OF3 ────────────────────────────────────────────
USE_MSA_SERVER       = True
USE_TEMPLATES        = False
NUM_ROLLOUT_STEPS    = None
NUM_DIFFUSION_SAMPLES = 1

# ── Confidence baseline ────────────────────────────
CONFIDENCE_PTM_CUTOFF     = 0.6
CONFIDENCE_PLDDT_CUTOFF   = 70.0
CONFIDENCE_RANKING_CUTOFF = 0.4
CONFIDENCE_RG_MAX         = 2.5
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True

# ── Fallback ───────────────────────────────────────
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True

# ── Autostop ───────────────────────────────────────
AUTOSTOP_V_MAGNITUDE     = 1.0
AUTOSTOP_N_STEPS         = 5000
AUTOSTOP_BACK_OFF        = 2
AUTOSTOP_FALLBACK_LEVELS = (0, 1, 4, 9)

# ── Converter checkpoint ──────────────────────────
DRIVE_DIR = "/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa"
CHECKPOINT_SELECTION = "best_bf_r"

# ── Hangi deneyleri çalıştır ──────────────────────
RUN_EXPERIMENTS = [
    "exp1_warmup",
    "exp2_rg_filter",
    "exp3_max_rejected",
    "exp4_alpha_decay",
    "exp5_pae_cutoff",
    # "exp6_consensus",      # K=3 gerekir, pahalı
    "exp7_contact_recon",
    "exp8_contact_of3",
]

## 3. Converter + PDB + OF3 yükleme

In [ ]:
# ════════════════════════════════════════════════════
#  LOAD CONVERTER + PDB + OF3
# ════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

import json as _json
import importlib
import numpy as np
import torch

# Force reload src
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score

# ── Checkpoint ──────────────────────────────────────
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded from {CHECKPOINT_PATH}')

# ── PDB fetch ───────────────────────────────────────
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

def fetch_ca(pdb_id, chain_id):
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence

initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
target_ca, _ = fetch_ca(TARGET_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'Initial: {INITIAL_PDB} chain {CHAIN_ID}, N={N}')
print(f'Target:  {TARGET_PDB} chain {CHAIN_ID}, N={target_ca.shape[0]}')
print(f'Initial RMSD to target: {compute_rmsd(initial_ca, target_ca):.2f} Å')
print(f'Initial TM-score: {tm_score(initial_ca, target_ca):.4f}')

# ── OF3 query + load ───────────────────────────────
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)

_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{
                'molecule_type': 'protein',
                'chain_ids': [CHAIN_ID],
                'sequence': sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion

diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')
# ── Autostop StructureContext ───────────────────────
from src.autostop_adapter import StructureContext

_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print(f'StructureContext: N={structure_ctx.struct.N} residues, '
      f'{structure_ctx.struct.n_atoms} heavy atoms')

print(f'OF3 diffusion ready.')

## 4. Baseline config oluşturucu

In [ ]:
# ════════════════════════════════════════════════════
#  BASELINE CONFIG + EXPERIMENT DEFINITIONS
# ════════════════════════════════════════════════════
import copy
import time
from dataclasses import asdict

def make_baseline():
    """Baseline ModeDriveConfig for grid search."""
    return ModeDriveConfig(
        n_steps=N_STEPS,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=Z_MIXING_ALPHA,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        confidence_ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
        confidence_plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
        confidence_ranking_cutoff=CONFIDENCE_RANKING_CUTOFF,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=CHAIN_ID,
    )


def run_one(overrides: dict, label: str = ""):
    """Run pipeline with overrides, return (result, step_metrics, wall_time)."""
    cfg = make_baseline()
    for k, v in overrides.items():
        setattr(cfg, k, v)
    
    pipe = ModeDrivePipeline(
        converter=converter,
        config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )
    
    print(f'\n{"="*70}')
    print(f'  {label}  |  {overrides}')
    print(f'{"="*70}')
    
    t0 = time.time()
    result = pipe.run(
        initial_ca.clone(), zij_trunk.clone(),
        verbose=True,
        target_coords=target_ca,
    )
    wall = time.time() - t0
    
    # Extract metrics
    metrics = []
    for i, sr in enumerate(result.step_results):
        m = {
            'step': i + 1,
            'rmsd': sr.rmsd,
            'ptm': sr.ptm,
            'plddt_mean': float(sr.plddt.mean()) if sr.plddt is not None else None,
            'ranking': sr.ranking_score,
            'rg_ratio': sr.rg_ratio,
            'contact_recon': sr.contact_recon,
            'contact_of3': sr.contact_of3,
            'mean_pae': sr.mean_pae,
            'has_clash': sr.has_clash,
            'consensus': sr.consensus_score,
            'fallback_level': sr.fallback_level,
            'rejected': sr.rejected,
            'df_used': sr.df_used,
            'alpha_used': sr.alpha_used,
        }
        metrics.append(m)
    
    accepted = sum(1 for sr in result.step_results if not sr.rejected)
    rejected = sum(1 for sr in result.step_results if sr.rejected)
    max_fb = max((sr.fallback_level for sr in result.step_results), default=0)
    final_rmsd = result.step_results[-1].rmsd if result.step_results else 0
    final_tm = tm_score(result.step_results[-1].new_ca, target_ca) if result.step_results else 0
    
    print(f'\n  => {accepted}/{result.total_steps} accepted, '
          f'max_fb={max_fb}, RMSD={final_rmsd:.2f}, TM={final_tm:.4f}, '
          f'wall={wall:.0f}s')
    
    return {
        'label': label,
        'overrides': overrides,
        'total_steps': result.total_steps,
        'accepted': accepted,
        'rejected': rejected,
        'max_fb': max_fb,
        'final_rmsd': final_rmsd,
        'final_tm': final_tm,
        'wall_s': wall,
        'step_metrics': metrics,
        'result': result,
    }


# Store all results
ALL_RESULTS = {}
print('Helpers ready.')

## 5. Deney 1: Warmup etkisi

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 1: WARMUP — ilk N step'te gevşek pTM/ranking cutoff
#  Baseline: pTM=0.6, ranking=0.4
#  Warmup'ta pTM'yi 0.35-0.50 arası gevşetiyoruz.
#  Mantık: ilk step'lerde yapı henüz OF3'e yabancı,
#  pTM düşük çıkması normal. Warmup bunu tolere eder.
# ════════════════════════════════════════════════════
exp1_configs = [
    {"confidence_warmup_steps": 0},                                                          # baseline (warmup yok)
    {"confidence_warmup_steps": 3, "confidence_warmup_ptm_cutoff": 0.50, "confidence_warmup_ranking_cutoff": 0.35},  # hafif
    {"confidence_warmup_steps": 5, "confidence_warmup_ptm_cutoff": 0.40, "confidence_warmup_ranking_cutoff": 0.30},  # orta
    {"confidence_warmup_steps": 8, "confidence_warmup_ptm_cutoff": 0.35, "confidence_warmup_ranking_cutoff": 0.25},  # agresif
    {"confidence_warmup_steps": 10, "confidence_warmup_ptm_cutoff": 0.30, "confidence_warmup_ranking_cutoff": 0.20}, # çok agresif
]

ALL_RESULTS['exp1_warmup'] = []
for i, ov in enumerate(exp1_configs):
    r = run_one(ov, label=f'exp1_warmup[{i}]')
    ALL_RESULTS['exp1_warmup'].append(r)

## 6. Deney 2: Rg filtresi

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 2: Rg FILTER — yapı patlaması / sıkışması tespiti
#  Rg_ratio = Rg_gözlenen / Rg_beklenen (2.2 * N^0.38)
#  1.0 = ideal, >2.5 = patlamış, <0.3 = aşırı sıkışmış
#  Üst sınırı sıkılaştırarak yapı kalitesini zorluyoruz.
# ════════════════════════════════════════════════════
exp2_configs = [
    {"confidence_rg_max": 999.0, "confidence_rg_min": 0.0},   # disabled (baseline)
    {"confidence_rg_max": 3.0,  "confidence_rg_min": 0.3},    # gevşek
    {"confidence_rg_max": 2.5,  "confidence_rg_min": 0.3},    # orta (default)
    {"confidence_rg_max": 2.0,  "confidence_rg_min": 0.4},    # sıkı
    {"confidence_rg_max": 1.8,  "confidence_rg_min": 0.5},    # çok sıkı
]

ALL_RESULTS['exp2_rg_filter'] = []
for i, ov in enumerate(exp2_configs):
    r = run_one(ov, label=f'exp2_rg[{i}]')
    ALL_RESULTS['exp2_rg_filter'].append(r)

## 7. Deney 3: Max consecutive rejected

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 3: STALL PREVENTION — max consecutive rejected
#  0 = sınırsız (cascade failure riski)
#  N ardışık rejected step sonrası pipeline durur.
#  Düşük = erken dur (compute tasarrufu), yüksek = daha toleranslı.
# ════════════════════════════════════════════════════
exp3_configs = [
    {"max_consecutive_rejected": 0},    # sınırsız (baseline — 30 step hep rejected olabilir)
    {"max_consecutive_rejected": 3},    # agresif durma
    {"max_consecutive_rejected": 5},    # orta
    {"max_consecutive_rejected": 8},    # toleranslı
    {"max_consecutive_rejected": 10},   # çok toleranslı
]

ALL_RESULTS['exp3_max_rejected'] = []
for i, ov in enumerate(exp3_configs):
    r = run_one(ov, label=f'exp3_stall[{i}]')
    ALL_RESULTS['exp3_max_rejected'].append(r)

## 8. Deney 4: Alpha decay on rejection

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 4: ALPHA DECAY — rejected sonrası alpha kademeli düşür
#  alpha=0.7 baseline. Her rejected step'te alpha *= decay.
#  Mantık: z_mixing çok agresif → OF3 bozuluyor → alpha'yı
#  düşürerek trunk z'ye daha yakın kalıyoruz.
#  Başarılı step'te alpha orijinal değerine döner.
# ════════════════════════════════════════════════════
exp4_configs = [
    {"rejected_alpha_decay": 1.0},    # decay yok (baseline)
    {"rejected_alpha_decay": 0.9},    # yavaş decay: 0.7→0.63→0.57→...
    {"rejected_alpha_decay": 0.8},    # orta: 0.7→0.56→0.45→0.36→...
    {"rejected_alpha_decay": 0.7},    # hızlı: 0.7→0.49→0.34→0.24→...
    {"rejected_alpha_decay": 0.5},    # çok hızlı: 0.7→0.35→0.18→...
]

ALL_RESULTS['exp4_alpha_decay'] = []
for i, ov in enumerate(exp4_configs):
    r = run_one(ov, label=f'exp4_decay[{i}]')
    ALL_RESULTS['exp4_alpha_decay'].append(r)

## 9. Deney 5: PAE cutoff

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 5: PAE CUTOFF — mean PAE max threshold (Å)
#  PAE = pairwise aligned error. AF3 max 32Å.
#  mean_pae düşük = iyi (residue çiftleri arasında
#  tahmin doğruluğu yüksek).
#  pTM zaten PAE'den türetiliyor ama bilgi kaybediyor.
#  mean_pae direkt ölçüm — en güçlü V2 metriği.
# ════════════════════════════════════════════════════
exp5_configs = [
    {"confidence_mean_pae_cutoff": None},    # disabled (baseline)
    {"confidence_mean_pae_cutoff": 25.0},    # çok gevşek
    {"confidence_mean_pae_cutoff": 20.0},    # gevşek
    {"confidence_mean_pae_cutoff": 15.0},    # orta
    {"confidence_mean_pae_cutoff": 10.0},    # sıkı
    {"confidence_mean_pae_cutoff": 8.0},     # çok sıkı
]

ALL_RESULTS['exp5_pae_cutoff'] = []
for i, ov in enumerate(exp5_configs):
    r = run_one(ov, label=f'exp5_pae[{i}]')
    ALL_RESULTS['exp5_pae_cutoff'].append(r)

## 10. Deney 6: K=3 sample consensus (opsiyonel, pahalı)

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 6: CONSENSUS — K=3 sample tutarlılığı
#  ⚠️ 3x daha fazla OF3 call → 3x süre
#  Çalıştırmadan önce diffusion_fn'ı K=3 ile yeniden yükle!
# ════════════════════════════════════════════════════

# Uncomment to run:
# # K=3 diffusion yeniden yükle
# diffusion_fn_k3, _ = load_of3_diffusion(
#     query_json=_query_path,
#     device='cuda',
#     use_msa_server=USE_MSA_SERVER,
#     num_samples=3,
# )
# 
# exp6_configs = [
#     {"num_diffusion_samples": 3, "confidence_consensus_cutoff": None},
#     {"num_diffusion_samples": 3, "confidence_consensus_cutoff": 0.3},
#     {"num_diffusion_samples": 3, "confidence_consensus_cutoff": 0.5},
# ]
# 
# ALL_RESULTS['exp6_consensus'] = []
# for i, ov in enumerate(exp6_configs):
#     # Temporarily swap diffusion_fn
#     r = run_one(ov, label=f'exp6_cons[{i}]')
#     ALL_RESULTS['exp6_consensus'].append(r)

print('Exp 6 skipped (uncomment to run with K=3)')

## 11. Deney 7: Contact reconstruction cutoff

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 7: CONTACT RECON — displaced vs output contact
#  contact_recon = Pearson r(contact(displaced), contact(new_ca))
#  Yüksek = OF3 verilen displacement'ı korumuş.
#  Düşük = OF3 tamamen farklı bir yapı üretmiş.
# ════════════════════════════════════════════════════
exp7_configs = [
    {"confidence_contact_recon_cutoff": None},    # disabled (baseline)
    {"confidence_contact_recon_cutoff": 0.2},     # çok gevşek
    {"confidence_contact_recon_cutoff": 0.3},     # gevşek
    {"confidence_contact_recon_cutoff": 0.5},     # orta
    {"confidence_contact_recon_cutoff": 0.7},     # sıkı
]

ALL_RESULTS['exp7_contact_recon'] = []
for i, ov in enumerate(exp7_configs):
    r = run_one(ov, label=f'exp7_cR[{i}]')
    ALL_RESULTS['exp7_contact_recon'].append(r)

## 12. Deney 8: OF3 distogram contact cutoff

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 8: CONTACT OF3 — displaced vs OF3 distogram
#  contact_of3 = Pearson r(contact(displaced), of3_contact_probs)
#  OF3'ün kendi distogram çıktısı ile input contact'ın uyumu.
#  Yüksek = OF3 aynı fikirde, düşük = OF3 farklı yapı görmüş.
# ════════════════════════════════════════════════════
exp8_configs = [
    {"confidence_contact_of3_cutoff": None},    # disabled (baseline)
    {"confidence_contact_of3_cutoff": 0.2},     # gevşek
    {"confidence_contact_of3_cutoff": 0.3},     # orta
    {"confidence_contact_of3_cutoff": 0.5},     # sıkı
]

ALL_RESULTS['exp8_contact_of3'] = []
for i, ov in enumerate(exp8_configs):
    r = run_one(ov, label=f'exp8_cOF3[{i}]')
    ALL_RESULTS['exp8_contact_of3'].append(r)

## 13. Karşılaştırma tablosu

In [ ]:
# ════════════════════════════════════════════════════
#  COMPARISON TABLE
# ════════════════════════════════════════════════════
import pandas as pd

rows = []
for exp_name, results in ALL_RESULTS.items():
    for r in results:
        rows.append({
            'experiment': exp_name,
            'label': r['label'],
            'overrides': str(r['overrides']),
            'steps': r['total_steps'],
            'accepted': r['accepted'],
            'rejected': r['rejected'],
            'max_fb': r['max_fb'],
            'final_rmsd': round(r['final_rmsd'], 2),
            'final_tm': round(r['final_tm'], 4),
            'wall_s': round(r['wall_s'], 0),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Highlight best per experiment
print('\n── Best per experiment (by TM-score) ──')
for exp_name in df['experiment'].unique():
    sub = df[df['experiment'] == exp_name]
    best = sub.loc[sub['final_tm'].idxmax()]
    print(f"  {exp_name}: {best['label']} → TM={best['final_tm']:.4f}, "
          f"RMSD={best['final_rmsd']:.2f}, {best['accepted']}/{best['steps']} accepted")

## 14. Per-step metrik grafikleri

In [ ]:
# ════════════════════════════════════════════════════
#  METRIC PLOTS — per-step comparison
# ════════════════════════════════════════════════════
import matplotlib.pyplot as plt

def plot_experiment(exp_name, metric='rmsd', ylabel=None):
    """Plot one metric across all configs in an experiment."""
    if exp_name not in ALL_RESULTS:
        print(f'{exp_name} not found')
        return
    
    fig, ax = plt.subplots(figsize=(12, 5))
    for r in ALL_RESULTS[exp_name]:
        steps = [m['step'] for m in r['step_metrics']]
        vals = [m.get(metric) for m in r['step_metrics']]
        # Replace None with NaN for plotting
        vals = [v if v is not None else float('nan') for v in vals]
        ax.plot(steps, vals, 'o-', label=r['label'], alpha=0.8)
    
    ax.set_xlabel('Step')
    ax.set_ylabel(ylabel or metric)
    ax.set_title(f'{exp_name} — {metric}')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_reject_map(exp_name):
    """Heatmap: step vs config, colored by rejected/accepted."""
    if exp_name not in ALL_RESULTS:
        return
    
    results = ALL_RESULTS[exp_name]
    n_configs = len(results)
    max_steps = max(r['total_steps'] for r in results)
    
    fig, ax = plt.subplots(figsize=(14, max(3, n_configs * 0.6)))
    for ci, r in enumerate(results):
        for m in r['step_metrics']:
            color = 'red' if m['rejected'] else 'green'
            fb = m['fallback_level']
            ax.scatter(m['step'], ci, c=color, s=80, alpha=0.7,
                      edgecolors='black', linewidths=0.5)
            if fb > 0:
                ax.annotate(str(fb), (m['step'], ci), fontsize=6,
                           ha='center', va='center', color='white', fontweight='bold')
    
    ax.set_yticks(range(n_configs))
    ax.set_yticklabels([r['label'] for r in results], fontsize=8)
    ax.set_xlabel('Step')
    ax.set_title(f'{exp_name} — Accept/Reject Map (green=accept, red=reject, number=FB level)')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


# Plot for each completed experiment
for exp_name in ALL_RESULTS:
    plot_experiment(exp_name, 'rmsd', 'RMSD from initial (Å)')
    plot_experiment(exp_name, 'rg_ratio', 'Rg ratio')
    plot_experiment(exp_name, 'contact_recon', 'Contact Recon (Pearson r)')
    plot_experiment(exp_name, 'mean_pae', 'Mean PAE (Å)')
    plot_reject_map(exp_name)
    print()

## 15. Deney 9: Kombine (en iyi sonuçlardan)

In [ ]:
# ════════════════════════════════════════════════════
#  EXP 9: COMBINED — en iyi tekli sonuçları birleştir
#  Yukarıdaki deneylerin sonuçlarına göre doldur!
# ════════════════════════════════════════════════════

# Önerilen kombinasyonlar (sonuçlara göre güncelle):
combined_configs = [
    {
        # A: Conservative — sadece fiziksel filtreler + warmup
        "confidence_warmup_steps": 5,
        "confidence_warmup_ptm_cutoff": 0.40,
        "confidence_warmup_ranking_cutoff": 0.30,
        "confidence_rg_max": 2.5,
        "confidence_rg_min": 0.3,
        "max_consecutive_rejected": 5,
        "rejected_alpha_decay": 0.9,
    },
    {
        # B: Balanced — fiziksel + PAE gate
        "confidence_warmup_steps": 5,
        "confidence_warmup_ptm_cutoff": 0.40,
        "confidence_warmup_ranking_cutoff": 0.30,
        "confidence_rg_max": 2.5,
        "confidence_rg_min": 0.3,
        "max_consecutive_rejected": 5,
        "rejected_alpha_decay": 0.85,
        "confidence_mean_pae_cutoff": 15.0,
    },
    {
        # C: Aggressive — tüm V2 filtreleri açık
        "confidence_warmup_steps": 3,
        "confidence_warmup_ptm_cutoff": 0.50,
        "confidence_warmup_ranking_cutoff": 0.35,
        "confidence_rg_max": 2.0,
        "confidence_rg_min": 0.4,
        "max_consecutive_rejected": 3,
        "rejected_alpha_decay": 0.8,
        "confidence_mean_pae_cutoff": 10.0,
        "confidence_contact_recon_cutoff": 0.3,
    },
    {
        # D: PAE-only — pTM gevşet, PAE ana gate
        "confidence_ptm_cutoff": 0.3,
        "confidence_ranking_cutoff": 0.2,
        "confidence_warmup_steps": 5,
        "confidence_warmup_ptm_cutoff": 0.20,
        "confidence_warmup_ranking_cutoff": 0.15,
        "confidence_rg_max": 2.5,
        "max_consecutive_rejected": 5,
        "rejected_alpha_decay": 0.85,
        "confidence_mean_pae_cutoff": 12.0,
    },
]

ALL_RESULTS['exp9_combined'] = []
for i, ov in enumerate(combined_configs):
    r = run_one(ov, label=f'exp9_combo[{i}]')
    ALL_RESULTS['exp9_combined'].append(r)

# Final comparison
plot_experiment('exp9_combined', 'rmsd', 'RMSD from initial (Å)')
plot_reject_map('exp9_combined')

## 16. Sonuçları kaydet

In [ ]:
# ════════════════════════════════════════════════════
#  SAVE RESULTS TO DRIVE
# ════════════════════════════════════════════════════
import json
from datetime import datetime

save_dir = '/content/drive/MyDrive/ANM-openfold3/grid_search_v2'
os.makedirs(save_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = os.path.join(save_dir, f'results_{timestamp}.json')

# Serialize (remove non-serializable 'result' key)
save_data = {}
for exp_name, results in ALL_RESULTS.items():
    save_data[exp_name] = []
    for r in results:
        r_copy = {k: v for k, v in r.items() if k != 'result'}
        save_data[exp_name].append(r_copy)

with open(save_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)

print(f'Results saved to {save_path}')

# Also save comparison table
csv_path = os.path.join(save_dir, f'comparison_{timestamp}.csv')
df.to_csv(csv_path, index=False)
print(f'CSV saved to {csv_path}')